# Style and Sector ETF Rotation with bt

ETF rotation is a natural target-weight problem. This notebook constructs De-Time rotation weights and shows how they can be passed to the `bt` framework.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
for path in [ROOT / "src", ROOT / "examples"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from quant_trading.data import fetch_yahoo_prices, fetch_yahoo_ohlcv, data_audit_report, DEFAULT_UNIVERSES
from quant_trading.features import decompose_one_series, walkforward_decompose, build_feature_table
from quant_trading.signals import (
    trend_pullback_signals,
    residual_mean_reversion_signals,
    turtle_donchian_signals,
    pair_trading_weights,
    cross_sectional_rotation_weights,
    residual_stress_filter,
)
from quant_trading.backtest import backtest_weights, backtest_long_short_signals, summarize_returns

In [2]:
universe = DEFAULT_UNIVERSES["us_sector_etfs"]
prices = fetch_yahoo_prices(universe, start="2016-01-01", cache_dir=ROOT / "examples" / "quant_trading" / "data" / "cache")
features = walkforward_decompose(prices, method="STL", period=63, train_window=252, step=21)
weights = cross_sectional_rotation_weights(prices, features, top_n=3, vol_target=0.12, max_weight=0.40)
result = backtest_weights(prices, weights, fee_bps=1.0, slippage_bps=2.0)
result.stats_frame()

,value
total_return,1.418749
cagr,0.088950
volatility,0.125759
sharpe,0.740762
max_drawdown,-0.193621
calmar,0.459403
hit_rate,0.496172
average_turnover,0.064707
average_gross_exposure,0.852037
fee_bps,1.000000


In [3]:
weights.tail().style.format("{:.2%}")

Ticker,XLK,XLF,XLE,XLV,XLY,XLP,XLI,XLU,XLB,XLRE,XLC
Date,,,,,,,,,,,
2026-05-18 00:00:00,0.00%,0.00%,36.84%,0.00%,0.00%,0.00%,0.00%,36.84%,36.84%,0.00%,0.00%
2026-05-19 00:00:00,0.00%,0.00%,37.01%,0.00%,0.00%,0.00%,0.00%,37.01%,37.01%,0.00%,0.00%
2026-05-20 00:00:00,0.00%,0.00%,36.98%,0.00%,0.00%,0.00%,0.00%,36.98%,36.98%,0.00%,0.00%
2026-05-21 00:00:00,0.00%,0.00%,37.04%,0.00%,0.00%,0.00%,0.00%,37.04%,37.04%,0.00%,0.00%
2026-05-22 00:00:00,0.00%,0.00%,37.04%,0.00%,0.00%,0.00%,0.00%,37.04%,37.04%,0.00%,0.00%


## Optional: bt adapter

In [4]:
from quant_trading.frameworks import run_bt_target_weights

# bt_result = run_bt_target_weights(prices, weights, name="detime_sector_rotation")
# bt_result.display()